# Deploy Approved Model to SageMaker Real-Time Endpoint

## 1. Retrieve the Latest Approved Model Package

In [ ]:
import json
from pathlib import Path

import boto3
import numpy as np
import pandas as pd
import joblib
import xgboost as xgb
from scipy import sparse

import sagemaker
from sagemaker import get_execution_role
from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig
from sagemaker.predictor import Predictor


boto_session = boto3.Session()
region = boto_session.region_name

account = boto_session.client("sts").get_caller_identity()["Account"]
bucket = f"sagemaker-{region}-{account}"
endpoint_name = "carepath"

model_package_arn = (
    f"arn:aws:sagemaker:{region}:{account}:"
    "model-package/CarePath-Diabetes-Readmission/1"
)

session = sagemaker.Session(boto_session=boto_session)
sagemaker_client = boto_session.client("sagemaker")
runtime = boto_session.client("sagemaker-runtime")
role = get_execution_role()

package = sagemaker_client.describe_model_package(
    ModelPackageName=model_package_arn
)
assert package["ModelPackageStatus"] == "Completed"

containers = package["InferenceSpecification"]["Containers"]
assert len(containers) == 1, "Expected a single model container."
container = containers[0]

data_capture_s3_uri = (
    f"s3://{bucket}/monitoring/video-demo/datacapture"
)

artifact_dir = Path("./artifacts")

required_files = [
    "preprocessor.joblib",
    "xgboost-model",
    "evaluation.json",
    "diabetes_processed.csv",
]
for filename in required_files:
    assert (artifact_dir / filename).is_file(), f"Missing: {filename}"

print("Region:", region)
print("Endpoint:", endpoint_name)
print("Package:", model_package_arn)
print("Registry approval status:", package["ModelApprovalStatus"])
print("Data capture:", data_capture_s3_uri)

Region: us-east-1
Endpoint: carepath
Package: arn:aws:sagemaker:us-east-1:769592409248:model-package/CarePath-Diabetes-Readmission/1
Registry approval status: PendingManualApproval
Data capture: s3://sagemaker-us-east-1-769592409248/monitoring/video-demo/datacapture


## 2. Configure Real-Time Endpoint with Data Capture
Use **DataCaptureConfig** to record request and response payloads to S3 for **SageMaker Model Monitor** and **CloudWatch**

In [8]:
# Check whether this endpoint already exists.
existing = [
    item
    for page in sagemaker_client.get_paginator(
        "list_endpoints"
    ).paginate()
    for item in page["Endpoints"]
    if item["EndpointName"] == endpoint_name
]

if existing:
    status = existing[0]["EndpointStatus"]
    print("Existing endpoint status:", status)

    if status != "InService":
        raise RuntimeError(
            f"Endpoint is {status}. Do not create another endpoint. "
            "Check its status or failure reason in SageMaker."
        )

    predictor = Predictor(
        endpoint_name=endpoint_name,
        sagemaker_session=session,
    )

else:
    # Deploy the package's image and artifacts as a separate demo model.
    # This does not change the package's registry approval status.
    model = Model(
        name=endpoint_name,
        image_uri=container["Image"],
        model_data=container["ModelDataUrl"],
        env=container.get("Environment", {}),
        role=role,
        predictor_cls=Predictor,
        sagemaker_session=session,
    )

    data_capture_config = DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
        csv_content_types=["text/csv"],
        sagemaker_session=session,
    )

    print(f"Deploying {endpoint_name}...")

    predictor = model.deploy(
        initial_instance_count=1,
        instance_type="ml.m5.large",
        endpoint_name=endpoint_name,
        variant_name="AllTraffic",
        data_capture_config=data_capture_config,
        tags=[{"Key": "Purpose", "Value": "VideoDemo"}],
        wait=True,
    )

description = sagemaker_client.describe_endpoint(
    EndpointName=endpoint_name
)
print("Endpoint:", endpoint_name)
print("Status:", description["EndpointStatus"])

/opt/conda/lib/python3.12/site-packages/sagemaker/model.py:347: SageMakerV2DeprecationWarning: Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Model is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `ModelBuilder` (`from sagemaker.serve import ModelBuilder`).
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Deploying carepath...
------!

/opt/conda/lib/python3.12/site-packages/sagemaker/base_predictor.py:140: SageMakerV2DeprecationWarning: Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.
  warn_v2_deprecation(
Predictor is part of the SageMaker Python SDK v2, which is on path of deprecation. In v3, use `sagemaker.core.resources.Endpoint`.
See https://github.com/aws/sagemaker-python-sdk/blob/master/migration.md for the migration guide. Set SAGEMAKER_SUPPRESS_V2_WARNING=1 to silence this warning.


Endpoint: carepath
Status: InService


## 3. Real-Time Endpoint Testing

In [9]:
evaluation = json.loads(
    (artifact_dir / "evaluation.json").read_text()
)
feature_cols = evaluation["raw_model_features"]
threshold = float(evaluation["decision_threshold"])

preprocessor = joblib.load(
    artifact_dir / "preprocessor.joblib"
)

booster = xgb.Booster()
booster.load_model(artifact_dir / "xgboost-model")

data = pd.read_csv(
    artifact_dir / "diabetes_processed.csv",
    keep_default_na=False,
)
sample = data.sample(n=5, random_state=42)

transformed = preprocessor.transform(sample[feature_cols])
expected = booster.predict(xgb.DMatrix(transformed))


def to_endpoint_csv(matrix):
    # Preserve implicit sparse entries as missing, while retaining
    # explicitly stored zeros.
    if sparse.issparse(matrix):
        coo = matrix.tocoo()
        dense = np.full(coo.shape, np.nan)
        dense[coo.row, coo.col] = coo.data
    else:
        dense = np.asarray(matrix)

    return ",".join(format(float(v), ".17g") for v in dense[0])


endpoint_scores = []

for i in range(len(sample)):
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        TargetVariant="AllTraffic",
        ContentType="text/csv",
        Accept="text/csv",
        Body=to_endpoint_csv(transformed[i:i + 1]),
    )

    with response["Body"] as body:
        score = float(body.read().decode().strip())

    assert np.isfinite(score) and 0 <= score <= 1
    endpoint_scores.append(score)

results = pd.DataFrame({
    "request": range(1, len(sample) + 1),
    "local_score": expected,
    "endpoint_score": endpoint_scores,
    "flagged": np.asarray(endpoint_scores) >= threshold,
})

display(results.round(6))

assert np.allclose(
    expected,
    endpoint_scores,
    atol=1e-5,
    rtol=1e-4,
), "Endpoint predictions differ from local predictions."

print("PASS: endpoint predictions match the saved model.")

,request,local_score,endpoint_score,flagged
0,1,0.332167,0.332167,False
1,2,0.261091,0.261091,False
2,3,0.408289,0.408289,False
3,4,0.468984,0.468984,True
4,5,0.412335,0.412335,False


PASS: endpoint predictions match the saved model.


In [27]:
print("Example csv payload:")
to_endpoint_csv(transformed[0])

Example csv payload:


'nan,nan,nan,nan,nan,nan,nan,1,nan,nan,nan,1,nan,nan,nan,nan,nan,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,1,nan,nan,nan,nan,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,1,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,1,nan,nan,nan,nan,nan,1,nan,nan,nan,1,nan,1,nan,1,2.2094918042337062,1.2650400618491768,-0.78359149355191204,0.49298781441683032,-0.28952220611426549,-0.20467583036304168,-0.49849657035350059,-1.2456552402101602'